# Generative AI Model Selection & Setup

### Model Selection & Setup


In this system, the Gemini API is integrated to support the generation of natural language explanations related to depression detection.  
The API is initialized using a secure API key stored in a .env file and loaded through environment variables to ensure that sensitive information is not exposed in the code.

The model **gemini-2.5-flash** was selected due to its ability to generate clear, context-aware, and human-readable responses.  
This is particularly important for a system that deals with sensitive topics such as mental health, where explanations must be understandable, supportive, and appropriately phrased.

The selected model provides a balance between response quality and system reliability, making it suitable for both development and testing.  
It allows the system to transform model predictions into meaningful explanations that can help users better understand their condition, while maintaining consistent performance

### API Key Management

The API key is stored in a local .env file to ensure that sensitive information is not exposed in the source code or shared in the repository.

The .env file contains environment variables, such as GEMINI_API_KEY, which are loaded at runtime using the python-dotenv library.  
This allows the system to securely access the API key without hardcoding it into the notebook.

The .env file is excluded from version control using .gitignore, ensuring that it is not uploaded to GitHub.  
Instead, a template file ( `.env.example`) can be shared with team members to guide them in setting up their own local environment variables.

This approach improves security and follows best practices for handling confidential data.

In [1]:
%pip install python-dotenv google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", gemini_api_key is not None)

client = genai.Client(api_key=gemini_api_key)

print("Gemini client loaded successfully")

API key loaded: True
Gemini client loaded successfully


## Prompt Template Design Documentation

The goal of the prompt engineering part is to design different prompt styles that convert the supervised model prediction into clear student lifestyle advice. The prediction output is based on the `Depression` target, where the student is classified as either at risk of depression or not at risk.

The prompts use the most relevant student lifestyle and academic features:

- CGPA
- Sleep Duration
- Study Hours
- Social Media Hours
- Physical Activity
- Stress Level
- Prediction Result

Four different prompt templates were designed to compare how prompt structure affects the quality of the generated advice.

## Prompt Templates

In [3]:
# ==============================
# Prompt Templates
# ==============================

def format_features(case):
    return f"""
CGPA: {case['cgpa']}
Sleep Duration: {case['sleep_duration']} hours
Study Hours: {case['study_hours']} hours
Social Media Hours: {case['social_media_hours']} hours
Physical Activity: {case['physical_activity']} hours
Stress Level: {case['stress_level']}
Prediction Result: {case['prediction']}
"""


template_1_basic = """
You are an assistant that provides simple student lifestyle advice.

Student information:
{features}

Based on the prediction result, explain the student's situation and give simple advice.

Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.
Keep the response clear and easy to understand.
"""


template_2_structured = """
You are a mental health and academic advisor.

Student information:
{features}

Provide:
1. Explanation
2. Risk factors
3. Lifestyle recommendations
4. Study balance advice

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.
"""


template_3_personalized = """
You are a supportive student coach.

Student information:
{features}

Give:
- Personalized advice
- Daily action steps
- Encouragement

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.
"""


template_4_analytical = """
You are a data-driven advisor.

Student information:
{features}

Explain:
- Why this prediction happened
- Which factors affected it
- What should be improved first

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.
"""

## Test Cases

In [4]:
test_cases = [
    {
        "case_id": "Case 1",
        "cgpa": 2.1,
        "sleep_duration": 4,
        "study_hours": 10,
        "social_media_hours": 5,
        "physical_activity": 0,
        "stress_level": 9,
        "prediction": "At risk of depression"
    },
    {
        "case_id": "Case 2",
        "cgpa": 3.5,
        "sleep_duration": 7,
        "study_hours": 5,
        "social_media_hours": 2,
        "physical_activity": 3,
        "stress_level": 3,
        "prediction": "Not at risk of depression"
    },
    {
        "case_id": "Case 3",
        "cgpa": 2.8,
        "sleep_duration": 5,
        "study_hours": 8,
        "social_media_hours": 4,
        "physical_activity": 1,
        "stress_level": 7,
        "prediction": "At risk of depression"
    }
]

## Prompt Preview

In [5]:
from IPython.display import display, Markdown

prompt_templates = {
    "Template 1": template_1_basic,
    "Template 2": template_2_structured,
    "Template 3": template_3_personalized,
    "Template 4": template_4_analytical
}

sample = format_features(test_cases[0])

for name, template in prompt_templates.items():
    display(Markdown(f"## {name}"))
    display(Markdown(template.format(features=sample)))

## Template 1


You are an assistant that provides simple student lifestyle advice.

Student information:

CGPA: 2.1
Sleep Duration: 4 hours
Study Hours: 10 hours
Social Media Hours: 5 hours
Physical Activity: 0 hours
Stress Level: 9
Prediction Result: At risk of depression


Based on the prediction result, explain the student's situation and give simple advice.

Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.
Keep the response clear and easy to understand.


## Template 2


You are a mental health and academic advisor.

Student information:

CGPA: 2.1
Sleep Duration: 4 hours
Study Hours: 10 hours
Social Media Hours: 5 hours
Physical Activity: 0 hours
Stress Level: 9
Prediction Result: At risk of depression


Provide:
1. Explanation
2. Risk factors
3. Lifestyle recommendations
4. Study balance advice

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.


## Template 3


You are a supportive student coach.

Student information:

CGPA: 2.1
Sleep Duration: 4 hours
Study Hours: 10 hours
Social Media Hours: 5 hours
Physical Activity: 0 hours
Stress Level: 9
Prediction Result: At risk of depression


Give:
- Personalized advice
- Daily action steps
- Encouragement

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.


## Template 4


You are a data-driven advisor.

Student information:

CGPA: 2.1
Sleep Duration: 4 hours
Study Hours: 10 hours
Social Media Hours: 5 hours
Physical Activity: 0 hours
Stress Level: 9
Prediction Result: At risk of depression


Explain:
- Why this prediction happened
- Which factors affected it
- What should be improved first

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.


# Implementation & API Integration Code 

In [6]:

def generate_ai_response(prompt):
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash", contents=prompt
        )
        return response.text
    except Exception as e:
        return f"Error: {e}"


## Prompt Execution

After designing the four prompt templates, each template was connected to the prepared test cases and executed using the Gemini API.

The purpose of this step is to generate sample outputs for each prompt template. These outputs will later be used by the output analysis to compare the templates based on relevance, clarity, personalization, completeness, and safety.



In [7]:
# ==============================
# Generate AI Outputs for All Templates (FINAL VERSION)
# ==============================

import pandas as pd
import os
import time

os.makedirs("Generative_AI/example_outputs", exist_ok=True)

results = []

def safe_generate(prompt, retries=3):
    for attempt in range(retries):
        response = generate_ai_response(prompt)

        # If success → return
        if not str(response).startswith("Error:"):
            return response

        # If quota error → wait and retry
        if "RESOURCE_EXHAUSTED" in str(response):
            print(f"Quota hit. Waiting 60s before retry ({attempt+1}/{retries})...")
            time.sleep(60)
        else:
            print("Other error:", response)
            return response

    return "Failed after retries"


for case in test_cases:
    features = format_features(case)

    for template_name, template in prompt_templates.items():
        final_prompt = template.format(features=features)

        print(f"Running: {case['case_id']} - {template_name}")

        ai_output = safe_generate(final_prompt)

        results.append({
            "case_id": case.get("case_id", "N/A"),
            "template_name": template_name,
            "prediction": case["prediction"],
            "prompt": final_prompt,
            "ai_output": ai_output
        })

        time.sleep(5)  # safe delay


outputs_df = pd.DataFrame(results)

outputs_df.to_csv(
    "Generative_AI/example_outputs/generated_ai_outputs.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done. Outputs saved.")
outputs_df

Running: Case 1 - Template 1
Running: Case 1 - Template 2
Running: Case 1 - Template 3
Running: Case 1 - Template 4
Running: Case 2 - Template 1
Running: Case 2 - Template 2
Running: Case 2 - Template 3
Running: Case 2 - Template 4
Running: Case 3 - Template 1
Running: Case 3 - Template 2
Running: Case 3 - Template 3
Running: Case 3 - Template 4
Done. Outputs saved.


,case_id,template_name,prediction,prompt,ai_output
0,Case 1,Template 1,At risk of depression,\nYou are an assistant that provides simple st...,Hi there! Thanks for reaching out. Let's look ...
1,Case 1,Template 2,At risk of depression,\nYou are a mental health and academic advisor...,Hello! Thank you for reaching out and sharing ...
2,Case 1,Template 3,At risk of depression,\nYou are a supportive student coach.\n\nStude...,Hey there! Thanks for reaching out and being o...
3,Case 1,Template 4,At risk of depression,\nYou are a data-driven advisor.\n\nStudent in...,"As your data-driven advisor, let's break down ..."
4,Case 2,Template 1,Not at risk of depression,\nYou are an assistant that provides simple st...,Hi there!\n\nIt looks like you're doing a fant...
5,Case 2,Template 2,Not at risk of depression,\nYou are a mental health and academic advisor...,Hello there! I'm here to provide some insights...
6,Case 2,Template 3,Not at risk of depression,\nYou are a supportive student coach.\n\nStude...,Hey there! I'm so glad to connect with you. Lo...
7,Case 2,Template 4,Not at risk of depression,\nYou are a data-driven advisor.\n\nStudent in...,"Based on the data provided, here's an analysis..."
8,Case 3,Template 1,At risk of depression,\nYou are an assistant that provides simple st...,Hey there! Thanks for sharing your information...
9,Case 3,Template 2,At risk of depression,\nYou are a mental health and academic advisor...,Hello! Thank you for coming to speak with me t...


In [8]:
# ==============================
# Save Outputs as Markdown
# ==============================

md_path = "Generative_AI/example_outputs/generated_ai_outputs.md"

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Generated AI Outputs\n\n")

    for _, row in outputs_df.iterrows():
        f.write(f"## {row['case_id']} - {row['template_name']}\n\n")
        f.write(f"**Prediction:** {row['prediction']}\n\n")
        f.write("### Prompt Used\n\n")
        f.write("```text\n")
        f.write(row["prompt"])
        f.write("\n```\n")
        f.write("### AI Output\n\n")
        f.write(row["ai_output"])
        f.write("\n\n---\n\n")

print(f"Saved markdown outputs to: {md_path}")

Saved markdown outputs to: Generative_AI/example_outputs/generated_ai_outputs.md


# Testing Framework & Output Comparison


## Evaluation Criteria

To systematically compare the four prompt templates, five evaluation criteria were established. Each criterion is scored on a four-point scale ranging from the lowest to the highest level of quality.

**Clarity and Readability** measures whether the advice is easy to understand for a non-technical audience and whether recommendations are expressed in clear, actionable language. The scale ranges from Poor (1) to Excellent (4).

**Relevance to Student Context** measures whether the recommendations address the specific student's situation based on their reported CGPA, sleep duration, stress level, and other features, and whether suggestions are grounded in the actual data rather than generic advice. The scale ranges from Low (1) to Highly Relevant (4).

**Personalization Level** measures whether the response feels tailored to the individual student and whether generic statements are avoided in favor of specific insights drawn from the student's profile. The scale ranges from Generic (1) to Highly Personalized (4).

**Completeness** measures whether the response addresses all key aspects including explanation, risk factors, recommendations, and study balance, and whether clear action steps or guidance are provided. The scale ranges from Incomplete (1) to Comprehensive (4).

**Safety and Responsibility** measures whether the response avoids claiming a clinical diagnosis, whether the language is supportive rather than alarming, and whether appropriate framing and disclaimers are included. The scale ranges from Unsafe (1) to Highly Safe (4).

## Testing Methodology

Three diverse student profiles were selected as test cases to cover different risk levels and lifestyle patterns.

Case 1 represents a high-risk student with very low sleep, a high stress level, no physical activity, and a low CGPA. Case 2 represents a low-risk student with adequate sleep, a low stress level, regular physical activity, and a strong CGPA. Case 3 represents a moderate-risk student with borderline metrics across multiple dimensions.

Each of the four templates was applied to each test case and responses were generated through the Gemini 2.5-Flash API. All outputs were collected in both CSV and Markdown formats. Manual review was then conducted using the five criteria described above, followed by scoring and comparative analysis across templates.

# Analysis: Qualitative and Quantitative Results

## Template Performance Summary

| Template | Clarity | Relevance | Personalization | Completeness | Safety | Best Suited For |
|---|---|---|---|---|---|---|
| T1: Basic | 3 / 4 | 3 / 4 | 2 / 4 | 2 / 4 | 4 / 4 | Quick, simple advice for general audiences |
| T2: Structured | 4 / 4 | 4 / 4 | 3 / 4 | 4 / 4 | 4 / 4 | Comprehensive reports with consistent formatting |
| T3: Personalized | 3 / 4 | 4 / 4 | 4 / 4 | 3 / 4 | 4 / 4 | Student engagement, motivation, and coaching |
| T4: Analytical | 4 / 4 | 4 / 4 | 3 / 4 | 3 / 4 | 4 / 4 | Data-driven explanations and interpretability |

## Key Findings by Template

### Template 1: Basic Lifestyle Advice

Template 1 produces concise and accessible responses that are easy for any user to follow. It demonstrates consistent safety compliance and avoids clinical language effectively. However, the analytical depth is limited and the level of personalization is low, which can make the advice feel generic when applied to students with very different risk profiles. The number of actionable recommendations is also lower compared to the other templates. This template is best suited for general awareness tools or lightweight feedback systems where brevity is prioritized.

### Template 2: Structured Mental Health and Academic Advice

Template 2 produces the most organized and comprehensive outputs across all test cases. The use of clearly labelled sections covering explanation, risk factors, lifestyle recommendations, and study balance advice makes it easy to read and to compare outputs across different student profiles. It performs equally well for both at-risk and not-at-risk predictions. The main limitation is that responses can become lengthy and may feel somewhat impersonal despite the depth of content. This template is best suited for institutional use, professional reports, and academic advisor workflows.

### Template 3: Personalized Student Coach

Template 3 produces the most engaging responses and achieves the highest personalization score. The inclusion of daily action steps and direct encouragement makes the advice feel achievable and motivating. It is most likely to drive actual behavior change in students who engage with it. The main limitations are that output length can vary considerably across cases and the analytical depth is lower than Templates 2 and 4. This template is best suited for student wellness applications, retention programs, and motivation-focused interventions.

### Template 4: Data-Driven Analytical Advice

Template 4 is the most effective at explaining why the prediction occurred and identifying which factors contributed most. It provides clear priority ordering for improvement areas and appeals to students who prefer a logical and evidence-based framing. The limitation is that the tone can feel detached and the language may not resonate with students who are seeking emotional support alongside practical guidance. This template is best suited for research contexts, model interpretability demonstrations, and students in analytical disciplines.

## Comparative Summary

Template 2 achieves the highest scores for both clarity and completeness, making it the most suitable for consistent, institution-wide deployment. Template 3 achieves the highest personalization score and is the most likely to produce outputs that students will act on. All four templates meet the safety requirement of avoiding clinical diagnosis language, and this standard is enforced consistently across all test cases. Template 1 is the most efficient option when processing speed or resource constraints are a priority.

# Best Prompt Selection and Justification

## Recommended Template: Template 2 — Structured Mental Health and Academic Advice

After evaluating all four templates across three student profiles using five criteria, Template 2 is selected as the primary template for this system. The following sections explain the reasoning behind this decision.

### Comprehensive and Organized Output

Template 2 is the only template that consistently addresses all four key dimensions: explanation of the prediction, identification of risk factors, lifestyle recommendations, and study balance advice. The structured format ensures that no important area is omitted regardless of the student profile, and the use of clear section headers makes it straightforward for both students and advisors to navigate the content.

### Suitability for Multiple Audiences

The output produced by Template 2 is accessible to students while remaining professional enough for use by academic advisors and institutional health services. This dual suitability makes it practical for integration into existing institutional workflows without requiring separate templates for different user types.

### Balance Between Empathy and Professionalism

Template 2 maintains a supportive tone without adopting the highly informal register of Template 3. This balance is appropriate for a sensitive domain such as mental health, where responses need to feel caring without crossing into territory that could be perceived as overclaiming or emotionally manipulative.

### Consistent Structure for System Integration

The predictable section structure of Template 2 outputs makes them easier to parse, store, and process programmatically. This is an important practical advantage when integrating the generative AI component into a larger system that may need to extract specific sections, log outputs to a database, or generate follow-up materials automatically.

### Evaluation Scores

Template 2 achieves the highest scores for clarity (4/4) and completeness (4/4), a strong score for safety (4/4), and a solid personalization score (3/4). The personalization score is slightly lower than Template 3, but this trade-off is acceptable given the gains in structure, depth, and consistency.

## Secondary Recommendations by Use Case

Template 3 is recommended as a secondary option for student wellness applications and retention programs where engagement and motivation are the primary goals. It is best used as an opt-in alternative to Template 2 rather than as a replacement.

Template 4 is recommended as a supplementary option for students who want to understand the specific factors driving their prediction. It can be offered alongside Template 2 for analytically oriented users or for research and interpretability purposes.

Template 1 is recommended as a fallback option when API resources are constrained or when a brief initial response is needed before a more detailed follow-up is generated.

## Recommended Deployment Architecture

The recommended approach is to use Template 2 as the default output for all predictions, with Template 3 and Template 4 available as optional views that students or advisors can request. Template 1 serves as the fallback when resource limits apply. This progressive disclosure model ensures that all users receive a complete and safe baseline response while allowing those who want more detail or a different framing to access it on demand.

# Integration Plan for Final System

## System Architecture Overview

The complete system is designed as a three-phase pipeline. In the first phase, student lifestyle and academic features are collected and preprocessed. These features include CGPA, sleep duration, study hours, social media hours, physical activity, and stress level. In the second phase, two parallel analytical components are applied: a supervised classification model that predicts depression risk, and an unsupervised clustering model that identifies student lifestyle profiles. In the third phase, the prediction output is passed to the Gemini API using Template 2 to generate a personalized, structured advice report.

## Implementation Stages

### Stage 1: Local Development and Testing

This stage is complete. All four notebooks have been developed and tested locally, prompt templates have been designed and evaluated, example outputs have been generated for three test cases, and safety guardrails have been validated across all templates.

The deliverables from this stage are Phase1_Data_Exploration.ipynb, Phase1_Supervised_Learning.ipynb, Phase2_Unsupervised_Learning.ipynb, and Phase2_Generative_AI.ipynb.

### Stage 2: API Integration and Backend Deployment

The backend service will load the trained Random Forest model, accept student feature inputs through an API, call the Gemini API using Template 2, and return a structured advice response. Three API endpoints are planned: a POST endpoint at /predict that accepts student data and returns the risk prediction along with generated advice, a GET endpoint at /history/{student_id} that retrieves the prediction history for a given student, and a POST endpoint at /feedback/{prediction_id} that collects outcome feedback for model improvement.

The database schema will include fields for student identifier, prediction date, predicted risk label, confidence score, cluster assignment, the full Gemini response stored as JSON, and advisor notes.

### Stage 3: User Interface

Three interface options are under consideration. A web portal would provide a student-facing dashboard displaying advice and progress tracking, an advisor interface for reviewing multiple students simultaneously, and PDF export functionality for formal reports. A mobile application would add push notifications for high-risk flagging, quick lifestyle input forms, and daily recommendation prompts. An LMS integration option would embed the system directly into platforms such as Canvas or Blackboard and enable automatic referrals to counseling services.

### Stage 4: Institutional Integration

The system is designed to be used by multiple stakeholder groups within an institution. Student health centers would use it to generate automatic referrals for at-risk students and to provide counselors with a structured summary of risk factors before appointments. Academic advisors would use it to identify at-risk students during advising sessions and to suggest workload adjustments where appropriate. Residential life staff would use cluster-level trend data to target peer support interventions. Research and analytics teams would use the logged outputs to track prediction accuracy over time and to evaluate which advice patterns correlate with positive student outcomes.

## Technical Requirements

The backend service will be implemented using Python with Flask or FastAPI and integrated with the Gemini API. PostgreSQL will be used for storing predictions and history, Redis will be used for model caching to reduce response latency, and structured logging will be implemented for error tracking and API quota monitoring.

On the security side, all API keys will be managed through environment variables with no hardcoding in source files. Data in transit will be encrypted using HTTPS and data at rest will be encrypted in the database. All access will be logged for audit purposes and data retention policies will be aligned with institutional requirements and applicable regulations.

Performance targets are set at under two seconds per prediction, 99.5 percent API availability, support for at least one thousand concurrent users, and approximately one gigabyte of storage per one hundred thousand predictions including history.

## Rollout Plan

The rollout is planned in three phases. During the first two months, a pilot deployment will be conducted with approximately one hundred volunteer students. The goals of this phase are to collect feedback on advice quality, monitor system stability, and refine prompts based on real usage patterns.

During months three and four, the system will be expanded to between five hundred and one thousand students. Advisors and counselors will be trained during this phase and integration testing with the student health center will be completed.

From month five onward, the system will be deployed university-wide with continuous monitoring, periodic prompt refinements, and annual model retraining using updated student data.

## Success Metrics

System performance will be measured by maintaining prediction accuracy above the Random Forest baseline of 74 percent, keeping API response times below two seconds, and achieving zero unplanned downtime incidents.

User adoption will be measured by the proportion of at-risk students who access advice (target: 30 percent or above), the proportion of advisors who use the system in student consultations (target: 20 percent or above), and average user satisfaction scores (target: 4 out of 5 or above).

Outcome impact will be assessed through measurable improvements in mental health services utilization, reductions in academic probation cases attributed to mental health factors, improved retention rates among flagged student populations, and improvements in student wellness survey scores.

Compliance will be verified by confirming zero unauthorized data access incidents, full safety guardrail compliance across all generated outputs, and adherence to all applicable data protection regulations.

# Ethical Considerations and Limitations

## Ethical Considerations

### Mental Health Sensitivity

Providing depression risk predictions without appropriate professional context carries a risk of causing harm to students who may misinterpret a risk indicator as a clinical finding. To address this, all four prompt templates explicitly instruct the model to treat predictions as risk indicators only and to avoid any language suggesting a clinical diagnosis. Responses are framed as supportive guidance rather than medical direction, and all templates include a recommendation to seek professional evaluation if symptoms persist or worsen. The system is positioned as a wellness screening tool, not a diagnostic instrument.

### Bias and Fairness

Machine learning models trained on historical data can perpetuate or amplify existing biases related to gender, socioeconomic background, culture, or institutional context. The features used in this system are behavioral and largely universal, which reduces some forms of demographic bias. The dataset covers one hundred thousand students, providing reasonable diversity, and the model uses class balancing to prevent majority-class dominance. However, the data originates from a single institutional context, which limits generalizability. Future work should include formal fairness audits across demographic subgroups and disaggregated performance metrics to identify any disparate impact in predictions or advice quality.

### Informed Consent and Transparency

Students should understand what data is being used, how predictions are generated, and what the output represents before interacting with the system. The system is designed with transparency in mind: the features used by the model are documented, the role of the Gemini API in generating advice is acknowledged, and no automated actions are taken based on risk scores without student awareness. Students should be provided with an informed consent notice at the point of data collection, with a clear option to decline participation or to request a human advisor instead of an AI-generated response.

### Confidentiality and Data Security

Mental health and academic data are highly sensitive and subject to data protection regulations such as FERPA. The system protects this data through environment variable management for API keys, HTTPS encryption for data in transit, database encryption for data at rest, and audit logging for all access events. Access is restricted to authorized advisors and counselors only. An incident response plan and annual security assessments are required before institutional deployment.

### Human Oversight and Autonomy

The system is designed to support human decision-making, not to replace it. AI-generated advice is intended to be reviewed by a qualified advisor before being shared with students in formal settings. No automatic referrals, academic holds, or other consequential actions are triggered by the prediction alone. Students always have access to a human counselor as an alternative. The recommended workflow places a human advisor between the AI output and any formal intervention.

### Equity of Access

Technology-based interventions can create unequal outcomes if they are only accessible to students with reliable internet access, high digital literacy, or compatible devices. The system should be designed to meet web accessibility standards, function on low-bandwidth connections, and be accompanied by non-digital alternatives such as paper forms or telephone check-ins. Access should be provided as an institutional service with no cost to students.

### Risk of Stigmatization

Labelling students as at risk, even with careful framing, can contribute to self-stigma or social stigma if predictions are not handled with discretion. To mitigate this, predictions are kept confidential and shared only with the consenting student and their assigned advisors. All communications frame the prediction as an early alert and a starting point for positive change rather than a fixed label. Advice focuses on modifiable behaviors and emphasizes that risk factors can be reduced through small, consistent changes.

---

## System Limitations

### Data Limitations

The model uses a limited set of engineered features derived from self-reported student data. Important factors such as family circumstances, socioeconomic pressures, prior mental health history, and medication use are not captured. All data represents a single point in time and does not reflect changes in a student's situation over weeks or months. Self-reported values for sleep duration, study hours, and stress level may be inaccurate due to recall bias or social desirability effects. The dataset originates from a single university context, which limits the extent to which findings can be generalized to other institutions or cultural settings.

### Model Limitations

The best performing model achieves 74.65 percent accuracy, which means approximately one in four predictions is incorrect. The recall for the at-risk class is 60.7 percent, indicating that the model misses approximately 40 percent of students who are genuinely at risk. This conservative behavior is partly a consequence of the class imbalance in the dataset, where only around 10 percent of students are labeled as depressed. The model is trained on static cross-sectional data and cannot predict whether a student's risk level is increasing or decreasing over time. No longitudinal validation has been conducted to confirm whether predicted risk corresponds to actual clinical outcomes months later.

### Generative AI Limitations

The Gemini API is non-deterministic, meaning that the same input may produce different outputs across separate calls. The model may occasionally generate advice that sounds plausible but is not grounded in the student's actual data. Output quality is sensitive to the exact wording of the prompt, and small changes to the template can produce noticeably different responses. API usage is subject to cost and quota limits, which may affect scalability during peak usage periods. The language model underlying the API reflects biases present in its training data, which may affect how advice is framed for students from different cultural backgrounds.

### Clustering Limitations

The KMeans clustering model assumes spherical, evenly sized clusters, which may not accurately reflect the natural groupings in student lifestyle data. The choice of three clusters was guided by the elbow method, which involves a degree of subjective judgment. DBSCAN identified all students as belonging to a single cluster in testing, suggesting that the data does not contain clearly separated high-density regions. Cluster assignments are descriptive and should not be interpreted as predictive of depression risk on their own.

### Integration and Adoption Challenges

Deploying the system in an institutional setting requires buy-in from multiple departments including health services, academic advising, and IT infrastructure. Privacy regulations may constrain how data is stored and shared across systems. Advisors may be resistant to adopting AI-generated summaries if they perceive the tool as a replacement for personal engagement rather than a supplement to it. Students may be skeptical of AI-generated mental health advice and may not engage with the system without deliberate outreach and trust-building efforts.

### Ethical Limitations

The system focuses on individual behavioral factors such as sleep, exercise, and study habits. It does not address structural and systemic contributors to student mental health such as institutional workload expectations, financial pressures, or housing instability. Framing depression risk primarily in terms of personal lifestyle choices may inadvertently suggest that mental health is solely a matter of individual responsibility, which is an incomplete and potentially harmful perspective. Additionally, there is currently no mechanism to verify whether students follow the advice provided or to measure whether that advice leads to improved outcomes over time.

---

## Recommendations for Future Work

On the modelling side, incorporating longitudinal data would allow the system to detect changes in student risk levels over time and to provide more timely interventions. Formal fairness audits across demographic subgroups should be conducted before any large-scale institutional deployment, and biological or physiological markers could be explored as supplementary inputs if available through existing health services.

On the generative AI side, a multi-turn conversational interface could improve personalization by allowing the system to ask clarifying questions before generating advice. An outcome tracking mechanism would enable evaluation of whether different advice patterns lead to measurable improvements in student well-being. Systematic A/B testing of prompt variations in real deployment conditions would provide empirical guidance for further prompt refinement.

On the institutional side, a formal workflow connecting the system to student health centers and academic advising offices is needed before the tool can operate at scale. A secure data infrastructure with full compliance documentation should be established, and a structured feedback mechanism should be implemented so that advisors can flag inaccurate or unhelpful outputs for review.

On the research and validation side, a prospective study comparing predicted risk against actual clinical diagnoses over a six-month follow-up period would provide the strongest evidence of model validity. A randomized controlled trial comparing outcomes between students who receive AI-generated advice and a control group would help establish whether the system produces meaningful improvements in mental health or academic performance. Qualitative interviews with students would help determine whether the system's framing and tone feel supportive and appropriate across diverse student populations.

---

## Conclusion

This system demonstrates that a combination of supervised classification, unsupervised clustering, and generative AI can be used to produce personalized, structured, and safety-compliant guidance for students flagged as at risk of depression. The generative AI component successfully translates a model prediction into readable, actionable advice across diverse student profiles and prompt formats.

The system is designed to complement, not replace, existing mental health services. Human advisors and counselors remain the primary decision-makers in any intervention. The role of AI in this system is to reduce the administrative burden of identifying at-risk students and to ensure that every flagged student receives a consistent, well-structured starting point for further support. All final decisions regarding referrals, care plans, and interventions must involve qualified professionals who can account for context, nuance, and individual circumstances that no automated system can fully capture.